#### Ejercicio 1: Obtener una detección del logo en cada imagen sin falsos positivos

In [2]:
%matplotlib qt

import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import supervision as sv
import os

In [3]:
# Para este ejercicio vamos a realizarlo mediante el uso de descriptores con SIFT.
# Primero tomamos la imagen template y buscamos descriptores. 
# Tomamos ademas una imagen de ejemplo para probar como funciona. 

imgTemplate = cv.imread(r'/home/gaspi/CEIA_CV1_TP3/assets/template/pattern.png', cv.IMREAD_GRAYSCALE)
imgSample = cv.imread(r'/home/gaspi/CEIA_CV1_TP3/assets/images/coca_retro_2.png', cv.IMREAD_GRAYSCALE)

In [5]:

# Canny
img_edges = cv.Canny(imgSample, 50, 150)
template_edges = cv.Canny(imgTemplate, 50, 150)


In [50]:
# Funcion de binarización (a mano)
def global_threshold(image, thres_value, val_high, val_low):
    img = image.copy()
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            if image[i,j] > thres_value:
                img[i,j] = val_high
            else:
                img[i,j] = val_low
    return img

In [51]:
img_binary = global_threshold(imgSample, 200, 255, 0)
# invertimos template
template_edges = cv.bitwise_not(imgTemplate)

In [52]:
# Creamos el vector de características SIFT
sift = cv.SIFT_create()

# Y buscamos según el algoritmo...
kp1, des1 = sift.detectAndCompute(template_edges,None)
kp2, des2 = sift.detectAndCompute(img_binary,None)

print(len(kp1))
print(len(kp2))

211
308


In [53]:
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
search_params = dict(checks = 50)
flann = cv.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des1,des2,k=2)

# Guardar todos las buenas coincidencias según la verificación de Ratio de Lowe
good = []
for m,n in matches:
    if m.distance < 0.7*n.distance:
        good.append(m)
print(len(good))

17


In [54]:
# https://docs.opencv.org/4.x/d9/dab/tutorial_homography.html

MIN_MATCH_COUNT = 8

if len(good) > MIN_MATCH_COUNT:
    src_pts = np.float32([ kp1[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
    dst_pts = np.float32([ kp2[m.trainIdx].pt for m in good ]).reshape(-1,1,2)
    M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
    matchesMask = mask.ravel().tolist()
    h,w = template_edges.shape
    pts = np.float32([ [0,0],[0,h-1],[w-1,h-1],[w-1,0] ]).reshape(-1,1,2)
    dst = cv.perspectiveTransform(pts, M)
    img2 = cv.polylines(img_binary, [np.int32(dst)], True, 255, 3, cv.LINE_AA)
else:
    print( "No se encontraron suficientes coincidencias - {}/{}".format(len(good), MIN_MATCH_COUNT) )
    matchesMask = None

print(mask)

[[1]
 [1]
 [1]
 [0]
 [1]
 [1]
 [1]
 [0]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [0]
 [0]]


In [55]:
draw_params = dict(matchColor = (0,255,0), # dibujar coincidencias en verde
                   singlePointColor = None,
                   matchesMask = matchesMask, # dibujar solo inliers
                   flags = 2)
img3 = cv.drawMatches(template_edges, kp1, img_binary, kp2, good, None, **draw_params)
sv.plot_image(img3)

In [48]:
# Ahora aplicamos el pipeline para todas las imagenes en images
path = r'/home/gaspi/CEIA_CV1_TP3/assets/images'
images = [
    os.path.join(path, f)
    for f in os.listdir(path)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
]

THRESH = 7 
MIN_MATCH_COUNT = 4

template_edges = cv.Canny(imgTemplate, 50, 150)
kp1, des1 = sift.detectAndCompute(template_edges,None)

for img_path in images:

    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
    img_edges = cv.Canny(img, 50, 150)

    kp2, des2 = sift.detectAndCompute(img_edges, None)

    if des2 is None:
        print(f"{os.path.basename(img_path)}: ❌ Sin descriptores")
        continue

    matches = flann.knnMatch(des1, des2, k=2)

    good = []
    for m, n in matches:
        if m.distance < 0.7 * n.distance:
            good.append(m)

    # ---- CALCULAR SCORE PRIMERO ----
    if len(good) >= MIN_MATCH_COUNT:

        src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)

        M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)

        if mask is not None:
            score = int(mask.sum())
        else:
            score = 0
    else:
        score = 0

    # ---- DIBUJAR MATCHES ----
    draw_matches = cv.drawMatches(
        imgTemplate, kp1,
        img, kp2,
        good, None,
        flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    # ---- VISUALIZACIÓN (AHORA SÍ) ----
    import matplotlib.pyplot as plt

    plt.figure(figsize=(12,6))
    plt.imshow(draw_matches, cmap='gray')
    plt.title(f"{os.path.basename(img_path)} - score: {score} - matches: {len(good)}")
    plt.axis('off')
    plt.show()

    # ---- DECISIÓN ----
    if score > THRESH:
        print(f"{os.path.basename(img_path)}: ✅ Hay logo ({score})")
    else:
        print(f"{os.path.basename(img_path)}: ❌ No hay logo ({score})")

no_coca_1.jpeg: ❌ No hay logo (0)
coca_logo_2.png: ✅ Hay logo (17)
coca_retro_1.png: ❌ No hay logo (0)
COCA-COLA-LOGO.jpg: ❌ No hay logo (4)
coca_multi.png: ❌ No hay logo (0)
coca_retro_2.png: ❌ No hay logo (0)
coca_logo_1.png: ✅ Hay logo (8)
logo_1.png: ✅ Hay logo (11)


In [13]:
# Observando los resultados, con este algoritmo tenemos cubiertas las imágenes logo_1, coca_logo_1, coca_logo_2
# Restan coca_retro_2.png, COCA-COLA-LOGO.jpg, coca_multi.png y coca_retro_1.png. 

# Veamos coca_retro_2.png para empezar
img = cv.imread('/home/gaspi/CEIA_CV1_TP3/assets/images/coca_retro_2.png', cv.IMREAD_GRAYSCALE)

kp, des = sift.detectAndCompute(img, None)

print("Keypoints:", len(kp))

kp_edges, des_edges = sift.detectAndCompute(cv.Canny(img,50,150), None)

print("Keypoints con Canny:", len(kp_edges))

# invertir canny
binary_edges = cv.Canny(img, 50, 150)
inverted_edges = cv.bitwise_not(binary_edges)


plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(img, cmap='gray')
plt.title("Original")
plt.axis('off')
plt.subplot(1,2,2)
plt.imshow(inverted_edges, cmap='gray')
plt.title("Canny")
plt.axis('off')
plt.show()


Keypoints: 1238
Keypoints con Canny: 913


In [14]:
# Podemos probar un algoritmo que saque descriptores con y sin canny y elija el mejor resultado. 

In [15]:
THRESH = 7 
MIN_MATCH_COUNT = 4

# Template con y sin Canny
template_edges = cv.Canny(imgTemplate, 50, 150)

kp1_edge, des1_edge = sift.detectAndCompute(template_edges, None)
kp1_raw, des1_raw   = sift.detectAndCompute(imgTemplate, None)

for img_path in images:

    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
    img_edges = cv.Canny(img, 50, 150)

    # =========================
    # 🔵 PIPELINE 1: CON CANNY
    # =========================
    kp2_e, des2_e = sift.detectAndCompute(img_edges, None)

    score_e = 0
    good_e = []

    if des2_e is not None:
        matches = flann.knnMatch(des1_edge, des2_e, k=2)

        for m, n in matches:
            if m.distance < 0.7 * n.distance:
                good_e.append(m)

        if len(good_e) >= MIN_MATCH_COUNT:
            src_pts = np.float32([kp1_edge[m.queryIdx].pt for m in good_e]).reshape(-1,1,2)
            dst_pts = np.float32([kp2_e[m.trainIdx].pt for m in good_e]).reshape(-1,1,2)

            M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
            if mask is not None:
                score_e = int(mask.sum())

    # =========================
    # 🟢 PIPELINE 2: SIN CANNY
    # =========================
    kp2_r, des2_r = sift.detectAndCompute(img, None)

    score_r = 0
    good_r = []

    if des2_r is not None:
        matches = flann.knnMatch(des1_raw, des2_r, k=2)

        for m, n in matches:
            if m.distance < 0.7 * n.distance:
                good_r.append(m)

        if len(good_r) >= MIN_MATCH_COUNT:
            src_pts = np.float32([kp1_raw[m.queryIdx].pt for m in good_r]).reshape(-1,1,2)
            dst_pts = np.float32([kp2_r[m.trainIdx].pt for m in good_r]).reshape(-1,1,2)

            M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
            if mask is not None:
                score_r = int(mask.sum())

    # =========================
    # 🏆 ELEGIR MEJOR RESULTADO
    # =========================
    if score_e > score_r:
        score = score_e
        good = good_e
        kp1_use, kp2_use = kp1_edge, kp2_e
        img_use, template_use = img_edges, template_edges
        mode = "CANNY"
    else:
        score = score_r
        good = good_r
        kp1_use, kp2_use = kp1_raw, kp2_r
        img_use, template_use = img, imgTemplate
        mode = "RAW"

    # =========================
    # 🎨 VISUALIZACIÓN
    # =========================
    draw_matches = cv.drawMatches(
        template_use, kp1_use,
        img_use, kp2_use,
        good, None,
        flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    import matplotlib.pyplot as plt

    plt.figure(figsize=(12,6))
    plt.imshow(draw_matches, cmap='gray')
    plt.title(f"{os.path.basename(img_path)} | {mode} | score: {score} | matches: {len(good)}")
    plt.axis('off')
    plt.show()

    # =========================
    # 🧾 DECISIÓN FINAL
    # =========================
    if score > THRESH:
        print(f"{os.path.basename(img_path)}: ✅ Hay logo ({score}) [{mode}]")
    else:
        print(f"{os.path.basename(img_path)}: ❌ No hay logo ({score}) [{mode}]")

no_coca_1.jpeg: ❌ No hay logo (4) [RAW]
coca_logo_2.png: ✅ Hay logo (17) [CANNY]
coca_retro_1.png: ✅ Hay logo (16) [RAW]
COCA-COLA-LOGO.jpg: ❌ No hay logo (5) [RAW]
coca_multi.png: ❌ No hay logo (0) [RAW]
coca_retro_2.png: ❌ No hay logo (0) [RAW]
coca_logo_1.png: ✅ Hay logo (8) [CANNY]
logo_1.png: ✅ Hay logo (11) [CANNY]


In [16]:
template_inv = cv.bitwise_not(imgTemplate)

plt.figure(figsize=(12,6))
plt.imshow(template_inv, cmap='gray')
plt.title(f"score: {score} | matches: {len(good)}")
plt.axis('off')
plt.show()

In [17]:
THRESH = 7 
MIN_MATCH_COUNT = 4

# Template con y sin Canny
template_inv = cv.bitwise_not(imgTemplate)
template_edges = cv.Canny(template_inv, 50, 150)

kp1_edge, des1_edge = sift.detectAndCompute(template_edges, None)
kp1_raw, des1_raw   = sift.detectAndCompute(imgTemplate, None)

for img_path in images:

    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
    img_edges = cv.Canny(img, 50, 150)

    # =========================
    # 🔵 PIPELINE 1: CON CANNY
    # =========================
    kp2_e, des2_e = sift.detectAndCompute(img_edges, None)

    score_e = 0
    good_e = []

    if des2_e is not None:
        matches = flann.knnMatch(des1_edge, des2_e, k=2)

        for m, n in matches:
            if m.distance < 0.7 * n.distance:
                good_e.append(m)

        if len(good_e) >= MIN_MATCH_COUNT:
            src_pts = np.float32([kp1_edge[m.queryIdx].pt for m in good_e]).reshape(-1,1,2)
            dst_pts = np.float32([kp2_e[m.trainIdx].pt for m in good_e]).reshape(-1,1,2)

            M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
            if mask is not None:
                score_e = int(mask.sum())

    # =========================
    # 🟢 PIPELINE 2: SIN CANNY
    # =========================
    kp2_r, des2_r = sift.detectAndCompute(img, None)

    score_r = 0
    good_r = []

    if des2_r is not None:
        matches = flann.knnMatch(des1_raw, des2_r, k=2)

        for m, n in matches:
            if m.distance < 0.7 * n.distance:
                good_r.append(m)

        if len(good_r) >= MIN_MATCH_COUNT:
            src_pts = np.float32([kp1_raw[m.queryIdx].pt for m in good_r]).reshape(-1,1,2)
            dst_pts = np.float32([kp2_r[m.trainIdx].pt for m in good_r]).reshape(-1,1,2)

            M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
            if mask is not None:
                score_r = int(mask.sum())

    # =========================
    # 🏆 ELEGIR MEJOR RESULTADO
    # =========================
    if score_e > score_r:
        score = score_e
        good = good_e
        kp1_use, kp2_use = kp1_edge, kp2_e
        img_use, template_use = img_edges, template_edges
        mode = "CANNY"
    else:
        score = score_r
        good = good_r
        kp1_use, kp2_use = kp1_raw, kp2_r
        img_use, template_use = img, imgTemplate
        mode = "RAW"

    # =========================
    # 🎨 VISUALIZACIÓN
    # =========================
    draw_matches = cv.drawMatches(
        template_use, kp1_use,
        img_use, kp2_use,
        good, None,
        flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    import matplotlib.pyplot as plt

    plt.figure(figsize=(12,6))
    plt.imshow(draw_matches, cmap='gray')
    plt.title(f"{os.path.basename(img_path)} | {mode} | score: {score} | matches: {len(good)}")
    plt.axis('off')
    plt.show()

    # =========================
    # 🧾 DECISIÓN FINAL
    # =========================
    if score > THRESH:
        print(f"{os.path.basename(img_path)}: ✅ Hay logo ({score}) [{mode}]")
    else:
        print(f"{os.path.basename(img_path)}: ❌ No hay logo ({score}) [{mode}]")

no_coca_1.jpeg: ❌ No hay logo (4) [RAW]
coca_logo_2.png: ✅ Hay logo (17) [CANNY]
coca_retro_1.png: ✅ Hay logo (17) [RAW]
COCA-COLA-LOGO.jpg: ❌ No hay logo (5) [RAW]
coca_multi.png: ❌ No hay logo (0) [RAW]
coca_retro_2.png: ❌ No hay logo (0) [RAW]
coca_logo_1.png: ✅ Hay logo (8) [CANNY]
logo_1.png: ✅ Hay logo (11) [CANNY]


In [18]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, color
from skimage.filters import threshold_sauvola

# --- Leer imagen ---
img = cv.imread('/home/gaspi/CEIA_CV1_TP3/assets/images/coca_retro_2.png', cv.IMREAD_GRAYSCALE)

# --- OTSU ---
_, img_bin = cv.threshold(img, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
img_bin_inv = cv.bitwise_not(img_bin)

# --- TEMPLATE ---
_, template_bin = cv.threshold(imgTemplate, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
template_bin_inv = cv.bitwise_not(template_bin)

# --- SAUVOLA ---
window_size = 15
k = 0.2
gray_img = img.astype(float) / 255
thresh_img = threshold_sauvola(gray_img, window_size=window_size, k=k)
img_sauvola = ((gray_img > thresh_img) * 255).astype(np.uint8)
img_sauvola_inv = cv.bitwise_not(img_sauvola)

# --- TEMPLATE SAUVOLA ---
gray_template = imgTemplate.astype(float) / 255
thresh_template = threshold_sauvola(gray_template, window_size=window_size, k=k)
template_sauvola = ((gray_template > thresh_template) * 255).astype(np.uint8)
template_sauvola_inv = cv.bitwise_not(template_sauvola)

# --- Variantes ---
variants_img = [img_bin, img_bin_inv, img_sauvola, img_sauvola_inv]
variants_tpl = [template_bin, template_bin_inv, template_sauvola, template_sauvola_inv]

results = []

for im_idx, im in enumerate(variants_img):
    for tpl_idx, tpl in enumerate(variants_tpl):

        kp1, des1 = sift.detectAndCompute(tpl, None)
        kp2, des2 = sift.detectAndCompute(im, None)

        if des1 is None or des2 is None:
            results.append((im_idx, tpl_idx, 0, [], [], None, None))
            continue

        matches = flann.knnMatch(des1, des2, k=2)

        good = [m for m, n in matches if m.distance < 0.75 * n.distance]

        if len(good) >= 4:

            src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
            dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)

            M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 10.0)

            if mask is not None:
                score = int(mask.sum())
                inliers = [good[i] for i in range(len(good)) if mask[i]]
            else:
                score = 0
                inliers = []
        else:
            score = 0
            inliers = []

        img_all = cv.drawMatches(
            tpl, kp1,
            im, kp2,
            good, None,
            matchColor=(255,0,0),
            flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
        )

        img_inliers = cv.drawMatches(
            tpl, kp1,
            im, kp2,
            inliers, None,
            matchColor=(0,255,0),
            flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
        )

        results.append((im_idx, tpl_idx, score, good, inliers, img_all, img_inliers))

# Encontrar el mejor
best_result = max(results, key=lambda x: x[2])
best_score, best_good, best_inliers, best_img_all, best_img_inliers = best_result[2:]

# =========================
# 📊 RESULTADOS
# =========================
print("Best score (inliers):", best_score)
print("Total matches:", len(best_good))
print("Inliers:", len(best_inliers))

if len(best_good) > 0:
    print("Inlier ratio:", len(best_inliers) / len(best_good))

# =========================
# 🎨 VISUALIZACIÓN DE TODAS LAS VARIANTES
# =========================
fig, axes = plt.subplots(4, 4, figsize=(20, 20))
variant_names_img = ['OTSU', 'OTSU_INV', 'SAUVOLA', 'SAUVOLA_INV']
variant_names_tpl = ['OTSU', 'OTSU_INV', 'SAUVOLA', 'SAUVOLA_INV']

for im_idx in range(4):
    for tpl_idx in range(4):
        result = next((r for r in results if r[0] == im_idx and r[1] == tpl_idx), None)
        if result and result[5] is not None:
            axes[im_idx, tpl_idx].imshow(result[5], cmap='gray')
            axes[im_idx, tpl_idx].set_title(f'Img: {variant_names_img[im_idx]}\nTpl: {variant_names_tpl[tpl_idx]}\nScore: {result[2]}')
        else:
            axes[im_idx, tpl_idx].set_title(f'Img: {variant_names_img[im_idx]}\nTpl: {variant_names_tpl[tpl_idx]}\nScore: 0')
        axes[im_idx, tpl_idx].axis('off')

plt.tight_layout()
plt.show()

# =========================
# 🎨 VISUALIZACIÓN DEL MEJOR
# =========================
if best_img_all is not None and best_img_inliers is not None:

    plt.figure(figsize=(14,6))

    plt.subplot(121)
    plt.imshow(best_img_all, cmap='gray')
    plt.title(f"ALL matches: {len(best_good)}")
    plt.axis('off')

    plt.subplot(122)
    plt.imshow(best_img_inliers, cmap='gray')
    plt.title(f"INLIERS (RANSAC): {len(best_inliers)}")
    plt.axis('off')

    plt.show()

Best score (inliers): 12
Total matches: 26
Inliers: 12
Inlier ratio: 0.46153846153846156


In [19]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import os

THRESH = 7 
MIN_MATCH_COUNT = 4

# =========================
# TEMPLATE PREPROCESADO
# =========================
template_inv = cv.bitwise_not(imgTemplate)

# 🔵 CANNY
template_edges = cv.Canny(template_inv, 50, 150)
kp1_edge, des1_edge = sift.detectAndCompute(template_edges, None)

# 🟢 RAW
kp1_raw, des1_raw = sift.detectAndCompute(imgTemplate, None)

# 🟣 OTSU
_, template_otsu = cv.threshold(imgTemplate, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
template_otsu_inv = cv.bitwise_not(template_otsu)

kp1_otsu, des1_otsu = sift.detectAndCompute(template_otsu, None)
kp1_otsu_inv, des1_otsu_inv = sift.detectAndCompute(template_otsu_inv, None)


# =========================
# LOOP IMÁGENES
# =========================
for img_path in images:

    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
    img_edges = cv.Canny(img, 50, 150)

    # =========================
    # 🔵 PIPELINE 1: CANNY
    # =========================
    kp2_e, des2_e = sift.detectAndCompute(img_edges, None)

    score_e = 0
    good_e = []

    if des2_e is not None and des1_edge is not None:
        matches = flann.knnMatch(des1_edge, des2_e, k=2)

        good_e = [m for m, n in matches if m.distance < 0.7 * n.distance]

        if len(good_e) >= MIN_MATCH_COUNT:
            src_pts = np.float32([kp1_edge[m.queryIdx].pt for m in good_e]).reshape(-1,1,2)
            dst_pts = np.float32([kp2_e[m.trainIdx].pt for m in good_e]).reshape(-1,1,2)

            M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
            if mask is not None:
                score_e = int(mask.sum())

    # =========================
    # 🟢 PIPELINE 2: RAW
    # =========================
    kp2_r, des2_r = sift.detectAndCompute(img, None)

    score_r = 0
    good_r = []

    if des2_r is not None and des1_raw is not None:
        matches = flann.knnMatch(des1_raw, des2_r, k=2)

        good_r = [m for m, n in matches if m.distance < 0.7 * n.distance]

        if len(good_r) >= MIN_MATCH_COUNT:
            src_pts = np.float32([kp1_raw[m.queryIdx].pt for m in good_r]).reshape(-1,1,2)
            dst_pts = np.float32([kp2_r[m.trainIdx].pt for m in good_r]).reshape(-1,1,2)

            M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
            if mask is not None:
                score_r = int(mask.sum())

    # =========================
    # 🟣 PIPELINE 3: OTSU
    # =========================
    _, img_otsu = cv.threshold(img, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    img_otsu_inv = cv.bitwise_not(img_otsu)

    variants_img = [img_otsu, img_otsu_inv]
    variants_tpl = [template_otsu, template_otsu_inv]

    score_o = 0
    good_o = []
    kp1_o_best, kp2_o_best = None, None
    img_o_best, tpl_o_best = None, None

    for tpl in variants_tpl:
        for im in variants_img:
            kp1_o, des1_o = sift.detectAndCompute(tpl, None)
            kp2_o, des2_o = sift.detectAndCompute(im, None)
            if des2_o is None or des1_o is None:
                continue

            matches = flann.knnMatch(des1_o, des2_o, k=2)
            good_tmp = [m for m, n in matches if m.distance < 0.75 * n.distance]

            if len(good_tmp) >= MIN_MATCH_COUNT:
                src_pts = np.float32([kp1_o[m.queryIdx].pt for m in good_tmp]).reshape(-1,1,2)
                dst_pts = np.float32([kp2_o[m.trainIdx].pt for m in good_tmp]).reshape(-1,1,2)

                M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 10.0)

                if mask is not None:
                    score_tmp = int(mask.sum())
                else:
                    score_tmp = 0
            else:
                score_tmp = 0

            # 🔥 selección correcta dentro de OTSU
            if score_tmp > score_o or (score_tmp == score_o and len(good_tmp) > len(good_o)):
                score_o = score_tmp
                good_o = good_tmp
                kp1_o_best, kp2_o_best = kp1_o, kp2_o
                img_o_best, tpl_o_best = im, tpl

    # =========================
    # 🏆 ELEGIR MEJOR RESULTADO (CORRECTO)
    # =========================
    score_e_total = score_e + 0.2 * len(good_e)
    score_r_total = score_r + 0.2 * len(good_r)
    score_o_total = score_o + 0.2 * len(good_o)

    scores = [score_e_total, score_r_total, score_o_total]
    best = np.argmax(scores)

    if best == 0:
        score = score_e
        good = good_e
        kp1_use, kp2_use = kp1_edge, kp2_e
        img_use, template_use = img_edges, template_edges
        mode = "CANNY"

    elif best == 1:
        score = score_r
        good = good_r
        kp1_use, kp2_use = kp1_raw, kp2_r
        img_use, template_use = img, imgTemplate
        mode = "RAW"

    else:
        score = score_o
        good = good_o
        kp1_use, kp2_use = kp1_o_best, kp2_o_best
        img_use, template_use = img_o_best, tpl_o_best
        mode = "OTSU"

    # =========================
    # 🎨 VISUALIZACIÓN
    # =========================
    draw_matches = cv.drawMatches(
        template_use, kp1_use,
        img_use, kp2_use,
        good, None,
        flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    plt.figure(figsize=(12,6))
    plt.imshow(draw_matches, cmap='gray')
    plt.title(f"{os.path.basename(img_path)} | {mode} | score: {score} | matches: {len(good)}")
    plt.axis('off')
    plt.show()

    # =========================
    # 🧾 DEBUG (MUY ÚTIL)
    # =========================
    print(f"\n{os.path.basename(img_path)}")

    # =========================
    # 🧾 DECISIÓN FINAL
    # =========================
    if score > THRESH:
        print(f"✅ Hay logo ({score}) [{mode}]")
    else:
        print(f"❌ No hay logo ({score}) [{mode}]")


no_coca_1.jpeg
❌ No hay logo (4) [RAW]

coca_logo_2.png
✅ Hay logo (37) [OTSU]

coca_retro_1.png
✅ Hay logo (27) [OTSU]

COCA-COLA-LOGO.jpg
✅ Hay logo (44) [OTSU]

coca_multi.png
❌ No hay logo (4) [OTSU]

coca_retro_2.png
❌ No hay logo (5) [OTSU]

coca_logo_1.png
✅ Hay logo (35) [OTSU]

logo_1.png
✅ Hay logo (11) [CANNY]


In [ ]:
# Con este pipeline solo resta detectar logos en coca_multi.png y coca_retro_2.png. 
# Se considera por un lado que coca_multi.png es un caso difícil para este mismo pipeline por la cantidad de logos y la variabilidad que presentan. 
# Por otro lado, coca_retro_2.png es un caso difícil por el bajo contraste y la gran cantidad de ruido.

#### Ejercicio 2: Plantear y validar un algoritmo para múltiples detecciones en la imagen coca_multi.png con el mismo témplate del ítem 1

In [20]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Leer imagen
img = cv.imread(r'/home/gaspi/CEIA_CV1_TP3/assets/images/coca_multi.png')
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

# Leer template
template = cv.imread(r'/home/gaspi/CEIA_CV1_TP3/assets/template/pattern.png', 0)
template_inv = cv.bitwise_not(template)

# Canny
img_edges = cv.Canny(img_gray, 50, 150)
template_edges = cv.Canny(template_inv, 50, 150)

method = cv.TM_CCOEFF_NORMED

# Construir pirámide gaussiana para el template
template_pyramid = []
current = template_edges
scales = []
for i in range(3):  # 3 niveles: 1.0, 0.5, 0.25
    template_pyramid.append(current)
    scales.append(1.0 / (2 ** i))
    current = cv.pyrDown(current)

for i, tpl in enumerate(template_pyramid):
    scale = scales[i]
    h, w = tpl.shape[:2]

    if h > img_edges.shape[0] or w > img_edges.shape[1]:
        continue

    # Matching
    res = cv.matchTemplate(img_edges, tpl, method)
    min_val, max_val, min_loc, max_loc = cv.minMaxLoc(res)

    top_left = max_loc

    # Dibujar detección
    img_salida = img.copy()
    bottom_right = (top_left[0] + w, top_left[1] + h)
    cv.rectangle(img_salida, top_left, bottom_right, (0,255,0), 2)

    # 🔥 Crear heatmap
    heatmap = cv.normalize(res, None, 0, 255, cv.NORM_MINMAX)
    heatmap = np.uint8(heatmap)
    heatmap_color = cv.applyColorMap(heatmap, cv.COLORMAP_JET)

    # 🔧 Redimensionar heatmap a tamaño original
    heatmap_resized = cv.resize(heatmap_color, (img.shape[1], img.shape[0]))

    # Mostrar lado a lado
    plt.figure(figsize=(10,4))

    plt.subplot(121)
    plt.imshow(cv.cvtColor(img_salida, cv.COLOR_BGR2RGB))
    plt.title(f'Detección | scale={scale:.2f}')
    plt.axis('off')

    plt.subplot(122)
    plt.imshow(cv.cvtColor(heatmap_resized, cv.COLOR_BGR2RGB))
    plt.title('Heatmap')
    plt.axis('off')

    plt.suptitle("Template Matching con Pirámide Gaussiana")
    plt.show()

In [21]:
threshold = 0.173  # configurable (0.7–0.9 típico)

# Construir pirámide gaussiana para el template
template_pyramid = []
current = template_edges
scales = []
for i in range(3):  # 3 niveles: 1.0, 0.5, 0.25
    template_pyramid.append(current)
    scales.append(1.0 / (2 ** i) if i > 0 else 1.0)
    current = cv.pyrDown(current)

img_salida = img.copy()

detecciones = []

for i, tpl in enumerate(template_pyramid):
    scale = scales[i]
    h, w = tpl.shape[:2]

    if h > img_edges.shape[0] or w > img_edges.shape[1]:
        continue

    res = cv.matchTemplate(img_edges, tpl, method)
    
    # buscamos todos los puntos donde supera el threshold
    loc = np.where(res >= threshold)

    for pt in zip(*loc[::-1]):
        detecciones.append((pt, w, h, res[pt[1], pt[0]]))

# -----------------------------
# FUNCIÓN IoU
# -----------------------------
def iou(boxA, boxB):
    (xA1, yA1, xA2, yA2) = boxA
    (xB1, yB1, xB2, yB2) = boxB

    xI1 = max(xA1, xB1)
    yI1 = max(yA1, yB1)
    xI2 = min(xA2, xB2)
    yI2 = min(yA2, yB2)

    interArea = max(0, xI2 - xI1) * max(0, yI2 - yI1)

    boxAArea = (xA2 - xA1) * (yA2 - yA1)
    boxBArea = (xB2 - xB1) * (yB2 - yB1)

    unionArea = boxAArea + boxBArea - interArea

    return interArea / unionArea if unionArea > 0 else 0

# -----------------------------
# NON-MAX SUPPRESSION (NMS)
# -----------------------------
boxes = []
for (pt, w, h, score) in detecciones:
    boxes.append((pt[0], pt[1], pt[0] + w, pt[1] + h, score))

# ordenar por score descendente
boxes = sorted(boxes, key=lambda x: x[4], reverse=True)

filtered = []
iou_threshold = 0.5  # ajustable

while boxes:
    best = boxes.pop(0)
    filtered.append(best)

    boxes = [
        box for box in boxes
        if iou(best[:4], box[:4]) < iou_threshold
    ]

# -----------------------------
# DIBUJAR RESULTADOS FILTRADOS
# -----------------------------
img_salida = img.copy()

for (x1, y1, x2, y2, score) in filtered:
    cv.rectangle(
        img_salida,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )

plt.figure(figsize=(8, 6))
plt.imshow(cv.cvtColor(img_salida, cv.COLOR_BGR2RGB))
plt.title(f"Detección multi-escala con Pirámide Gaussiana + NMS | threshold = {threshold}")
plt.axis("off")
plt.show()

print("Detecciones originales:", len(detecciones))
print("Detecciones filtradas:", len(filtered))

Detecciones originales: 61
Detecciones filtradas: 20


In [ ]:
# Tiene varios falsos positivos. 


#### Ejercicio 3: Generalizar el algoritmo del item 2 para todas las imágenes.

In [22]:
import os
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# CONFIG
# -----------------------------
input_folder = "/home/gaspi/CEIA_CV1_TP3/assets/images"
threshold = 0.2
iou_threshold = 0.5
method = cv.TM_CCOEFF_NORMED

# extensiones válidas
exts = (".jpg", ".jpeg", ".png", ".bmp")

# -----------------------------
# PIRÁMIDE GAUSSIANA TEMPLATE
# -----------------------------
template_pyramid = []
scales = []

current = template_edges.copy()

for i in range(3):  # 1.0, 0.5, 0.25
    scale = 1.0 / (2 ** i)
    template_pyramid.append(current)
    scales.append(scale)

    current = cv.pyrDown(current)


# -----------------------------
# LOOP DE IMÁGENES
# -----------------------------
for filename in os.listdir(input_folder):
    if not filename.lower().endswith(exts):
        continue

    print(f"Procesando: {filename}")

    img_path = os.path.join(input_folder, filename)
    img = cv.imread(img_path)

    if img is None:
        continue

    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    img_edges = cv.Canny(gray, 50, 150)

    detecciones = []

    # -----------------------------
    # MATCHING MULTI-ESCALA
    # -----------------------------
    for tpl, scale in zip(template_pyramid, scales):
        h, w = tpl.shape[:2]

        if h > img_edges.shape[0] or w > img_edges.shape[1]:
            continue

        res = cv.matchTemplate(img_edges, tpl, method)

        loc = np.where(res >= threshold)

        for pt in zip(*loc[::-1]):
            x, y = pt

            x1 = x
            y1 = y
            x2 = x + w
            y2 = y + h

            score = res[y, x]

            detecciones.append((x1, y1, x2, y2, score))

    # -----------------------------
    # NMS
    # -----------------------------
    detecciones = sorted(detecciones, key=lambda x: x[4], reverse=True)

    filtered = []
    while detecciones:
        best = detecciones.pop(0)
        filtered.append(best)

        detecciones = [
            box for box in detecciones
            if iou(best[:4], box[:4]) < iou_threshold
        ]

    # -----------------------------
    # DIBUJAR
    # -----------------------------
    img_out = img.copy()

    for (x1, y1, x2, y2, score) in filtered:
        cv.rectangle(img_out, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # -----------------------------
    # MOSTRAR
    # -----------------------------
    plt.figure(figsize=(8, 6))
    plt.imshow(cv.cvtColor(img_out, cv.COLOR_BGR2RGB))
    plt.title(f"{filename} | detecciones: {len(filtered)}")
    plt.axis("off")
    plt.show()

print("✔ Procesamiento completo")

Procesando: no_coca_1.jpeg
Procesando: coca_logo_2.png
Procesando: coca_retro_1.png
Procesando: COCA-COLA-LOGO.jpg
Procesando: coca_multi.png
Procesando: coca_retro_2.png
Procesando: coca_logo_1.png
Procesando: logo_1.png
✔ Procesamiento completo
